In [ ]:
import json
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Set fonts.
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Data directory.
base_dir = "paper/results/gemini-2.0-flash-001"

model = 'gpt-oss-120b'
efforts = ['low', 'medium', 'high']
effort_labels = ['Low', 'Medium', 'High']
target_levels = [1, 2, 3]  # Analyze only levels 1, 2, and 3.

# Store data as data[effort][level][context_length] = safe_rate.
data = defaultdict(lambda: defaultdict(dict))

for effort in efforts:
    condition_dir = f'{model}-{effort}'
    condition_path = os.path.join(base_dir, condition_dir)
    if not os.path.exists(condition_path):
        continue
    
    for filename in os.listdir(condition_path):
        if filename.startswith('eval_level_'):
            match = re.match(r'eval_level_(\d+)_(\d+)k_results_google-gemini-2.0-flash-lite-001.json', filename)
            if match:
                level = int(match.group(1))
                context_length = int(match.group(2))
                
                # Load only levels 1, 2, and 3.
                if level not in target_levels:
                    continue
                
                filepath = os.path.join(condition_path, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        result = json.load(f)
                        stats = result.get('statistics', {})
                        total = stats.get('total', 0)
                        score_dist = stats.get('score_distribution', {})
                        score_5_count = score_dist.get('5', 0)
                        
                        if total > 0:
                            safe_rate = (total - score_5_count) / total
                            data[effort][level][context_length] = safe_rate * 100
                        else:
                            data[effort][level][context_length] = 0.0
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")

# Collect all context lengths.
all_context_lengths = sorted(set(
    ctx for effort_data in data.values() 
    for level_data in effort_data.values() 
    for ctx in level_data.keys()
))

# ========== Reasoning token statistics ==========
import tiktoken

# Try tiktoken, falling back to a simple word count if unavailable.
try:
    enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text):
        return len(enc.encode(text)) if text else 0
except:
    def count_tokens(text):
        return len(text.split()) if text else 0

# Store reasoning token data.
reasoning_tokens = {effort: [] for effort in efforts}

for effort in efforts:
    condition_dir = f'{model}-{effort}'
    condition_path = os.path.join(base_dir, condition_dir)
    if not os.path.exists(condition_path):
        continue
    
    for filename in os.listdir(condition_path):
        if filename.startswith('level_') and filename.endswith('_results.json'):
            match = re.match(r'level_(\d+)_(\d+)k_results.json', filename)
            if match:
                level = int(match.group(1))
                if level not in target_levels:
                    continue
                
                filepath = os.path.join(condition_path, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        results = json.load(f)
                        for item in results:
                            if 'generated' in item and 'thinking' in item['generated']:
                                thinking = item['generated']['thinking']
                                if thinking:
                                    tokens = count_tokens(thinking)
                                    reasoning_tokens[effort].append(tokens)
                except:
                    pass

# Calculate the average reasoning token count.
avg_tokens = {effort: np.mean(reasoning_tokens[effort]) if reasoning_tokens[effort] else 0 
              for effort in efforts}

print(f"Model: {model}")
print(f"Target Levels: {target_levels}")
print(f"Context Lengths: {all_context_lengths}")
print(f"\nAverage Reasoning Tokens:")
for effort in efforts:
    print(f"  {effort.capitalize()}: {avg_tokens[effort]:.0f} tokens")
print(f"\nData loaded successfully!")


### Impact of Reasoning Effort

In [ ]:
# Create two plots: A line plot above and a bar plot with efficiency annotations below.
fig, axes = plt.subplots(2, 1, figsize=(10, 12))

# Set the global font size.
plt.rcParams.update({'font.size': 14})

# Define colors and markers.
effort_colors = {'low': '#3498db', 'medium': '#f39c12', 'high': '#e74c3c'}
effort_markers = {'low': 'o', 'medium': 's', 'high': '^'}
effort_linestyles = {'low': '--', 'medium': '-.', 'high': '-'}

# Use evenly spaced X-axis positions.
x_positions = np.arange(len(all_context_lengths))
ctx_to_pos = {ctx: i for i, ctx in enumerate(all_context_lengths)}

# ========== Upper plot: Combined line plot averaged across three levels ==========
ax1 = axes[0]

# Calculate the average safety ratio across all levels for each effort and context length.
avg_by_ctx = {effort: [] for effort in efforts}
for ctx in all_context_lengths:
    for effort in efforts:
        values = []
        for level in target_levels:
            if ctx in data[effort][level]:
                values.append(data[effort][level][ctx])
        avg_by_ctx[effort].append(np.mean(values) if values else 0)

# Calculate low-effort values as the baseline for improvement ratios.
low_avg = avg_by_ctx['low']

for effort in efforts:
    ax1.plot(x_positions, avg_by_ctx[effort],
            color=effort_colors[effort],
            marker=effort_markers[effort],
            markersize=10,
            linewidth=2.5,
            linestyle=effort_linestyles[effort],
            label=effort.capitalize(),
            markerfacecolor=effort_colors[effort],
            markeredgecolor='white',
            markeredgewidth=1.5)
    # Annotate medium- and high-effort points with their improvement factors relative to low effort.
    if effort in ['medium', 'high']:
        for i, (pos, sr) in enumerate(zip(x_positions, avg_by_ctx[effort])):
            low_val = avg_by_ctx['low'][i]
            if low_val > 0:
                ratio = sr / low_val
                offset_y = 8 if effort == 'high' else -15
                va = 'bottom' if effort == 'high' else 'top'
                ax1.annotate(f'{ratio:.2f}x', 
                           xy=(pos, sr),
                           xytext=(0, offset_y), textcoords="offset points",
                           ha='center', va=va,
                           fontsize=10, fontweight='bold',
                           color=effort_colors[effort])


# Annotate the gaps at the first and last positions.
for i in [0, len(x_positions)-1]:
    low_val = avg_by_ctx['low'][i]
    high_val = avg_by_ctx['high'][i]
    gap = high_val - low_val
    # Draw double-headed arrows.
    ax1.annotate('', xy=(i + 0.25, high_val), xytext=(i + 0.25, low_val),
                arrowprops=dict(arrowstyle='<->', color='#333333', lw=2))
                
    ax1.annotate(f'+{gap:.0f}', xy=(i + 0.2, low_val + 5),
                fontsize=14, fontweight='bold', color='#333333', ha='right', va='bottom')

ax1.set_xlabel('Context Length', fontsize=15, fontweight='bold')
ax1.set_ylabel('Safety Ratio (%)', fontsize=15, fontweight='bold')
ax1.set_title('Safety vs Context Length\n(Averaged across reasoning types)', fontsize=16, fontweight='bold')
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f'{ctx}k' for ctx in all_context_lengths], fontsize=12)
ax1.set_xlim(-0.5, len(all_context_lengths) - 0.5)
ax1.set_ylim(0, 100)
ax1.set_yticks([0, 20, 40, 60, 80, 100])
ax1.tick_params(axis='y', labelsize=12)
ax1.grid(True, alpha=0.5, linestyle='--', color='#444444')
ax1.legend(loc='lower left', fontsize=13, framealpha=0.9)

for spine in ax1.spines.values():
    spine.set_color('#333333')
    spine.set_linewidth(1.5)

# ========== Lower plot: Safety and token cost bars ==========
ax2 = axes[1]
ax2_twin = ax2.twinx()

# Calculate the overall average safety ratio for each effort.
overall_avg = []
for effort in efforts:
    all_values = []
    for level in target_levels:
        all_values.extend(data[effort][level].values())
    overall_avg.append(np.mean(all_values) if all_values else 0)

# Token data.
token_vals = [avg_tokens[e] for e in efforts]

x_pos = np.arange(len(efforts))
bar_width = 0.35

# Plot safety ratio bars on the left Y-axis.
bars1 = ax2.bar(x_pos - bar_width/2, overall_avg, bar_width, 
                label='Safety Ratio', color='#27ae60', edgecolor='white', linewidth=1.5)

# Plot reasoning token bars on the right Y-axis.
bars2 = ax2_twin.bar(x_pos + bar_width/2, token_vals, bar_width,
                     label='Reasoning Tokens', color='#9b59b6', edgecolor='white', linewidth=1.5, alpha=0.85)

# Add safety value labels.
for bar, sr in zip(bars1, overall_avg):
    ax2.annotate(f'{sr:.1f}%', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=13, fontweight='bold', color='#27ae60')

# Add token value labels.
for bar, tk in zip(bars2, token_vals):
    ax2_twin.annotate(f'{tk:.0f}', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=13, fontweight='bold', color='#9b59b6')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(effort_labels, fontsize=14)
ax2.set_xlabel('Reasoning Effort', fontsize=15, fontweight='bold')
ax2.set_ylabel('Safety Ratio (%)', fontsize=15, fontweight='bold', color='#27ae60')
ax2_twin.set_ylabel('Avg Reasoning Tokens', fontsize=15, fontweight='bold', color='#9b59b6')
ax2.set_title('More Thinking = Safer (but Costlier)', fontsize=16, fontweight='bold')
ax2.set_ylim(0, 100)
ax2_twin.set_ylim(0, max(token_vals) * 1.35)
ax2.tick_params(axis='y', labelcolor='#27ae60', labelsize=12)
ax2_twin.tick_params(axis='y', labelcolor='#9b59b6', labelsize=12)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

# Add a legend.
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=12, framealpha=0.9)

for spine in ax2.spines.values():
    spine.set_color('#333333')
    spine.set_linewidth(1.5)

# ========== Add marginal efficiency annotations to the lower plot ==========
# Calculate marginal efficiency: Safety improvement per 100 additional tokens.
marginal_safety = [overall_avg[1] - overall_avg[0], overall_avg[2] - overall_avg[1]]
marginal_tokens = [token_vals[1] - token_vals[0], token_vals[2] - token_vals[1]]
marginal_efficiency = [marginal_safety[i] / marginal_tokens[i] * 100 if marginal_tokens[i] > 0 else 0 
                       for i in range(2)]

# Draw curved arrows between bars and label efficiency, raising the start to avoid covering numbers.
y_offset = 8  # Offset for the arrow's starting position.
for i, (eff, ds, dt) in enumerate(zip(marginal_efficiency, marginal_safety, marginal_tokens)):
    # Arrow start and end points, above the numeric labels on the bars.
    x_start = x_pos[i] - bar_width/2
    x_end = x_pos[i+1] - bar_width/2
    y_start = overall_avg[i] + y_offset
    y_end = overall_avg[i+1] + y_offset
    
    # Draw curved arrows; larger absolute rad values produce greater curvature.
    ax2.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5,
                               connectionstyle='arc3,rad=-0.5'))
    
    # Label efficiency at the arrow midpoint, below the arrow to avoid the legend.
    mid_x = (x_start + x_end) / 2
    mid_y = (y_start + y_end) / 2 - 3
    ax2.annotate(f'+{eff:.1f}%/100tok', 
                xy=(mid_x, mid_y),
                ha='center', va='top', fontsize=10, fontweight='bold',
                color='#2c3e50',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.8))

plt.tight_layout()
plt.savefig(f'reasoning_effort_analysis_{model}.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\n图表已保存为 'reasoning_effort_analysis.png'")


In [ ]:
# Create two plots: A line plot on the left and a bar plot with efficiency annotations on the right.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Set the global font size.
plt.rcParams.update({'font.size': 14})

# Define colors and markers.
effort_colors = {'low': '#3498db', 'medium': '#f39c12', 'high': '#e74c3c'}
effort_markers = {'low': 'o', 'medium': 's', 'high': '^'}
effort_linestyles = {'low': '--', 'medium': '-.', 'high': '-'}

# Use evenly spaced X-axis positions.
x_positions = np.arange(len(all_context_lengths))
ctx_to_pos = {ctx: i for i, ctx in enumerate(all_context_lengths)}

# ========== Upper plot: Combined line plot averaged across three levels ==========
ax1 = axes[0]

# Calculate the average safety ratio across all levels for each effort and context length.
avg_by_ctx = {effort: [] for effort in efforts}
for ctx in all_context_lengths:
    for effort in efforts:
        values = []
        for level in target_levels:
            if ctx in data[effort][level]:
                values.append(data[effort][level][ctx])
        avg_by_ctx[effort].append(np.mean(values) if values else 0)

# Calculate low-effort values as the baseline for improvement ratios.
low_avg = avg_by_ctx['low']

for effort in efforts:
    ax1.plot(x_positions, avg_by_ctx[effort],
            color=effort_colors[effort],
            marker=effort_markers[effort],
            markersize=10,
            linewidth=2.5,
            linestyle=effort_linestyles[effort],
            label=effort.capitalize(),
            markerfacecolor=effort_colors[effort],
            markeredgecolor='white',
            markeredgewidth=1.5)
    # Annotate medium- and high-effort points with their improvement factors relative to low effort.
    if effort in ['medium', 'high']:
        for i, (pos, sr) in enumerate(zip(x_positions, avg_by_ctx[effort])):
            low_val = avg_by_ctx['low'][i]
            if low_val > 0:
                ratio = sr / low_val
                offset_y = 8 if effort == 'high' else -15
                va = 'bottom' if effort == 'high' else 'top'
                ax1.annotate(f'{ratio:.2f}x', 
                           xy=(pos, sr),
                           xytext=(0, offset_y), textcoords="offset points",
                           ha='center', va=va,
                           fontsize=10, fontweight='bold',
                           color=effort_colors[effort])


# Annotate the gaps at the first and last positions.
for i in [0, len(x_positions)-1]:
    low_val = avg_by_ctx['low'][i]
    high_val = avg_by_ctx['high'][i]
    gap = high_val - low_val
    # Draw double-headed arrows.
    ax1.annotate('', xy=(i + 0.25, high_val), xytext=(i + 0.25, low_val),
                arrowprops=dict(arrowstyle='<->', color='#333333', lw=2))
                
    ax1.annotate(f'+{gap:.0f}', xy=(i + 0.2, low_val + 5),
                fontsize=14, fontweight='bold', color='#333333', ha='right', va='bottom')

ax1.set_xlabel('Context Length', fontsize=15, fontweight='bold')
ax1.set_ylabel('Safety Ratio (%)', fontsize=15, fontweight='bold')
ax1.set_title('Safety vs Context Length\n(Averaged across reasoning types)', fontsize=16, fontweight='bold')
ax1.set_xticks(x_positions)
ax1.set_xticklabels([f'{ctx}k' for ctx in all_context_lengths], fontsize=12)
ax1.set_xlim(-0.5, len(all_context_lengths) - 0.5)
ax1.set_ylim(0, 100)
ax1.set_yticks([0, 20, 40, 60, 80, 100])
ax1.tick_params(axis='y', labelsize=12)
ax1.grid(True, alpha=0.5, linestyle='--', color='#444444')
ax1.legend(loc='lower left', fontsize=13, framealpha=0.9)

for spine in ax1.spines.values():
    spine.set_color('#333333')
    spine.set_linewidth(1.5)

# ========== Lower plot: Safety and token cost bars ==========
ax2 = axes[1]
ax2_twin = ax2.twinx()

# Calculate the overall average safety ratio for each effort.
overall_avg = []
for effort in efforts:
    all_values = []
    for level in target_levels:
        all_values.extend(data[effort][level].values())
    overall_avg.append(np.mean(all_values) if all_values else 0)

# Token data.
token_vals = [avg_tokens[e] for e in efforts]

x_pos = np.arange(len(efforts))
bar_width = 0.35

# Plot safety ratio bars on the left Y-axis.
bars1 = ax2.bar(x_pos - bar_width/2, overall_avg, bar_width, 
                label='Safety Ratio', color='#27ae60', edgecolor='white', linewidth=1.5)

# Plot reasoning token bars on the right Y-axis.
bars2 = ax2_twin.bar(x_pos + bar_width/2, token_vals, bar_width,
                     label='Reasoning Tokens', color='#9b59b6', edgecolor='white', linewidth=1.5, alpha=0.85)

# Add safety value labels.
for bar, sr in zip(bars1, overall_avg):
    ax2.annotate(f'{sr:.0f}%', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=13, fontweight='bold', color='#27ae60')

# Add token value labels.
for bar, tk in zip(bars2, token_vals):
    ax2_twin.annotate(f'{tk:.0f}', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=13, fontweight='bold', color='#9b59b6')

ax2.set_xticks(x_pos)
ax2.set_xticklabels(effort_labels, fontsize=14)
ax2.set_xlabel('Reasoning Effort', fontsize=15, fontweight='bold')
ax2.set_ylabel('Safety Ratio (%)', fontsize=15, fontweight='bold', color='#27ae60')
ax2_twin.set_ylabel('Avg Reasoning Tokens', fontsize=15, fontweight='bold', color='#9b59b6')
ax2.set_title('More Thinking = Safer (but Costlier)', fontsize=16, fontweight='bold')
ax2.set_ylim(0, 100)
ax2_twin.set_ylim(0, max(token_vals) * 1.35)
ax2.tick_params(axis='y', labelcolor='#27ae60', labelsize=12)
ax2_twin.tick_params(axis='y', labelcolor='#9b59b6', labelsize=12)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

# Add a legend.
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=12, framealpha=0.9)

for spine in ax2.spines.values():
    spine.set_color('#333333')
    spine.set_linewidth(1.5)

# ========== Add marginal efficiency annotations to the lower plot ==========
# Calculate marginal efficiency: Safety improvement per 100 additional tokens.
marginal_safety = [overall_avg[1] - overall_avg[0], overall_avg[2] - overall_avg[1]]
marginal_tokens = [token_vals[1] - token_vals[0], token_vals[2] - token_vals[1]]
marginal_efficiency = [marginal_safety[i] / marginal_tokens[i] * 100 if marginal_tokens[i] > 0 else 0 
                       for i in range(2)]

# Draw curved arrows between bars and label efficiency, raising the start to avoid covering numbers.
y_offset = 8  # Offset for the arrow's starting position.
for i, (eff, ds, dt) in enumerate(zip(marginal_efficiency, marginal_safety, marginal_tokens)):
    # Arrow start and end points, above the numeric labels on the bars.
    x_start = x_pos[i] - bar_width/2
    x_end = x_pos[i+1] - bar_width/2
    y_start = overall_avg[i] + y_offset
    y_end = overall_avg[i+1] + y_offset
    
    # Draw curved arrows; larger absolute rad values produce greater curvature.
    ax2.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5,
                               connectionstyle='arc3,rad=-0.5'))
    
    # Label efficiency at the arrow midpoint, below the arrow to avoid the legend.
    mid_x = (x_start + x_end) / 2
    mid_y = (y_start + y_end) / 2 - 3
    ax2.annotate(f'+{eff:.1f}%/100tok', 
                xy=(mid_x, mid_y),
                ha='center', va='top', fontsize=10, fontweight='bold',
                color='#2c3e50',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.8))

plt.tight_layout()
plt.savefig(f'reasoning_effort_analysis_{model}_raw.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\n图表已保存为 'reasoning_effort_analysis.png'")


### The impact of Reasoning Type

In [ ]:
# Show differences between reasoning types.
# More complex reasoning types (multi-hop) are harder for the model to recognize and have lower safety ratios.

fig, ax1 = plt.subplots(figsize=(10, 6))

# Level name mapping.
level_names = {1: 'Single-Hop', 2: 'Chain', 3: 'Multi-Hop'}

# Effort color mapping, using slightly darker colors.
effort_colors = {'low': '#A0C4D9', 'medium': '#F5C89A', 'high': '#D99990'}

# Set the global font size.
plt.rcParams.update({'font.size': 14})

# Calculate the average safety ratio for each level and effort.
avg_by_level = {level: [] for level in target_levels}
for level in target_levels:
    for effort in efforts:
        values = list(data[effort][level].values())
        avg_by_level[level].append(np.mean(values) if values else 0)

# ========== Comparison bars for low, medium, and high effort ==========

x_pos = np.arange(len(target_levels))
bar_width = 0.25

# Get the data.
low_vals = [avg_by_level[level][0] for level in target_levels]
med_vals = [avg_by_level[level][1] for level in target_levels]
high_vals = [avg_by_level[level][2] for level in target_levels]


# Low-effort bars: Light blue.
bars_low = ax1.bar(x_pos - bar_width, low_vals, bar_width,
                   label='Low', color=effort_colors['low'], 
                   edgecolor='white', linewidth=1.5)

# Medium-effort bars: Peach.
bars_med = ax1.bar(x_pos, med_vals, bar_width,
                   label='Medium', color=effort_colors['medium'],
                   edgecolor='white', linewidth=1.5)

# High-effort bars: Light red.
bars_high = ax1.bar(x_pos + bar_width, high_vals, bar_width,
                    label='High', color=effort_colors['high'],
                    edgecolor='white', linewidth=1.5)

# Add numeric labels in dark colors for readability.
for bar, val in zip(bars_low, low_vals):
    ax1.annotate(f'{val:.0f}%', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2E5B73')

for bar, val in zip(bars_med, med_vals):
    ax1.annotate(f'{val:.0f}%', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#A67D50')

for bar, val in zip(bars_high, high_vals):
    ax1.annotate(f'{val:.0f}%', 
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=12, fontweight='bold', color='#943D38')

# Add improvement arrows and percentage labels, all starting from low effort.
for i, level in enumerate(target_levels):
    low_v = low_vals[i]
    med_v = med_vals[i]
    high_v = high_vals[i]
    
    # Improvement from low to medium effort.
    if low_v > 0:
        imp_low_med = (med_v - low_v) / low_v * 100
    else:
        imp_low_med = 0
    
    # Improvement from low to high effort.
    if low_v > 0:
        imp_low_high = (high_v - low_v) / low_v * 100
    else:
        imp_low_high = 0
    
    # Use black arrows.
    arrow_color = '#333333'
    
    # Arrow 1: Low to medium effort.
    ax1.annotate('', xy=(x_pos[i], med_v - 2),
                xytext=(x_pos[i] - bar_width, low_v + 5),
                arrowprops=dict(arrowstyle='->', color=arrow_color, lw=1.5,
                               connectionstyle='arc3,rad=0.15'))
    # Place the label at the arrow midpoint.
    mid_x1 = x_pos[i] - bar_width/2
    mid_y1 = (low_v + med_v) / 2
    ax1.annotate(f'+{imp_low_med:.0f}%', 
                xy=(mid_x1, mid_y1),
                ha='center', va='center', fontsize=11, fontweight='bold',
                color=arrow_color,
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.8))
    
    # Arrow 2: Low to high effort.
    ax1.annotate('', xy=(x_pos[i] + bar_width, high_v - 2),
                xytext=(x_pos[i] - bar_width, low_v + 5),
                arrowprops=dict(arrowstyle='->', color=arrow_color, lw=1.5,
                               connectionstyle='arc3,rad=0.3'))
    # Place the label at the arrow midpoint.
    mid_x2 = x_pos[i] + 0.2
    mid_y2 = (low_v + high_v) / 2 + 5
    ax1.annotate(f'+{imp_low_high:.0f}%', 
                xy=(mid_x2, mid_y2),
                ha='center', va='center', fontsize=11, fontweight='bold',
                color=arrow_color,
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.8))

ax1.set_xticks(x_pos)
ax1.set_xticklabels([level_names[l] for l in target_levels], fontsize=14)
ax1.set_xlabel('Reasoning Type', fontsize=15, fontweight='bold')
ax1.set_ylabel('Avg Safety Ratio (%)', fontsize=15, fontweight='bold')
ax1.set_title('Low vs High Effort by Task Complexity', fontsize=16, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.set_yticks([0, 20, 40, 60, 80, 100])
ax1.tick_params(axis='y', labelsize=12)
ax1.grid(True, alpha=0.5, linestyle='--', color='#444444', axis='y')
ax1.legend(loc='best', fontsize=12, framealpha=0.9)

for spine in ax1.spines.values():
    spine.set_color('#333333')
    spine.set_linewidth(1.5)

plt.tight_layout()
# plt.savefig('reasoning_type_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig(f'reasoning_type_comparison_{model}.pdf', dpi=300, bbox_inches='tight')
plt.show()
